<a href="https://colab.research.google.com/github/msubham1504-svg/Quantum-error-correction-/blob/main/Error_Calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Step 1-Install the quantum-computing tools**

Copy this code into the first cell and press the ▶ Run button.


In [ ]:

%pip install qiskit qiskit-aer matplotlib


In [ ]:

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# Create one qubit and one classical bit
qc = QuantumCircuit(1, 1)

# Put the qubit into a superposition
qc.h(0)

# Measure the qubit
qc.measure(0, 0)

# Simulate 1000 measurements
simulator = AerSimulator()
result = simulator.run(qc, shots=1000).result()

print(result.get_counts())
print(qc.draw())

**Step 2 — Creating the first working test**

This first test constructs the Steane code's seven-qubit logical zero state. It uses the standard CSS construction from the classical [7,4,3] Hamming code.
Paste this into a new cell:

In [ ]:

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

# The 8 basis states of the Steane logical-zero state
codewords = [
    "0000000", "0011110",
    "0101101", "0110011",
    "1001011", "1010101",
    "1100110", "1111000"
]

# Prepare an equal superposition of these states
amplitudes = np.zeros(128, dtype=complex)

for word in codewords:
    amplitudes[int(word, 2)] = 1 / np.sqrt(8)

qc = QuantumCircuit(7, 7)
qc.initialize(amplitudes, range(7))
qc.measure(range(7), range(7))

print("Seven-qubit Steane logical-zero test")
print(qc.draw())

simulator = AerSimulator()
result = simulator.run(qc, shots=1000).result()
counts = result.get_counts()

print("\nMeasurement results:")
for word, count in sorted(counts.items()):
    print(word, ":", count)

print("\nNumber of distinct outcomes:", len(counts))

we see **eight distinct bitstrings**, each occurring roughly*125 times over 1,000 shots.* The counts will fluctuate randomly.

These are the eight basis states making up the logical zero state, \(|1\rangle\).

**Step-3: Verifying the six stabilizers**

Before building the correction circuit, To confirm that your encoded state is correct. This is an important test because errors in the stabilizer definitions can break the whole project.

In [ ]:

from qiskit.quantum_info import Statevector, Pauli
import numpy as np

# Recreate the encoded logical-zero state
codewords = [
    "0000000", "0011110",
    "0101101", "0110011",
    "1001011", "1010101",
    "1100110", "1111000"
]

amplitudes = np.zeros(128, dtype=complex)

for word in codewords:
    amplitudes[int(word, 2)] = 1 / np.sqrt(8)

logical_zero = Statevector(amplitudes)

# Three independent stabilizer supports
supports = ["0011110", "0101101", "1001011"]

print("Checking all six stabilizers...\n")

for support in supports:
    for gate in ["X", "Z"]:
        label = "".join(
            gate if bit == "1" else "I"
            for bit in support
        )

        expectation = np.real(
            logical_zero.expectation_value(Pauli(label))
        )

        print(
            f"{gate} stabilizer {support}: "
            f"expectation = {expectation:.1f}"
        )

print("\nExpected result: all six expectations equal +1")

All six expectation values should equal 1.0.

That means the state is a +1 eigenstate of each stabilizer, which is exactly what we expect for the logical-zero state.

**Step 4— Test whether the code can detect errors**

Now we'll inject an artificial error and see whether the stabilizers notice it.
Run this in another new cell:

In [ ]:

# Inject an X error on physical qubit q0
error_qubit = 0

error_label = "IIIIIIX"
noisy_state = logical_zero.evolve(Pauli(error_label))

print("Injected error: X on q0\n")

for support in supports:
    label = "".join(
        "Z" if bit == "1" else "I"
        for bit in support
    )

    value = np.real(
        noisy_state.expectation_value(Pauli(label))
    )

    syndrome_bit = "0" if value > 0 else "1"

    print(
        f"Stabilizer {support}: "
        f"value = {value:+.1f}, "
        f"syndrome bit = {syndrome_bit}"
    )

We got three syndrome bits, with the pattern 011 for this error under our chosen stabilizer order.

Why? An \(X\) error anticommutes with a \(Z\) stabilizer whenever the stabilizer acts on that qubit. The sign changes from \(+1\) to \(-1\), revealing the error without directly measuring the encoded quantum information.

**One important distinction:** this is an ideal mathematical test using statevectors, not yet a physical syndrome-measurement circuit. Our next stage will replace these direct expectation-value checks with ancilla qubits, measurements, and conditional corrections.

**Step 4 — Building the syndrome extraction and correction circuit**

In [ ]:

import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator

# -----------------------------------------
# 1. Prepare the Steane logical-zero state
# -----------------------------------------
codewords = [
    "0000000", "0011110",
    "0101101", "0110011",
    "1001011", "1010101",
    "1100110", "1111000"
]

amplitudes = np.zeros(128, dtype=complex)

for word in codewords:
    amplitudes[int(word, 2)] = 1 / np.sqrt(8)

# Seven data qubits + six ancilla qubits
data = QuantumRegister(7, "data")
anc = QuantumRegister(6, "anc")
sx = ClassicalRegister(3, "sx")
sz = ClassicalRegister(3, "sz")
out = ClassicalRegister(7, "out")

qc = QuantumCircuit(data, anc, sx, sz, out)

qc.initialize(amplitudes, list(data))

# -----------------------------------------
# 2. Deliberately introduce an X error on q0
# -----------------------------------------
qc.x(data[0])

# Stabilizer supports; string positions are q6...q0
supports = ["0011110", "0101101", "1001011"]

# -----------------------------------------
# 3. Measure Z-type stabilizers
# These detect X errors
# -----------------------------------------
for j, support in enumerate(supports):
    a = anc[j]

    for q in range(7):
        if support[6 - q] == "1":
            qc.cx(data[q], a)

    qc.measure(a, sx[j])

# -----------------------------------------
# 4. Measure X-type stabilizers
# These detect Z errors
# -----------------------------------------
for j, support in enumerate(supports):
    a = anc[3 + j]

    qc.h(a)

    for q in range(7):
        if support[6 - q] == "1":
            qc.cx(a, data[q])

    qc.h(a)
    qc.measure(a, sz[j])

# -----------------------------------------
# 5. Decode the syndrome and correct
# -----------------------------------------
for q in range(7):
    # Syndrome value for an X error on q
    x_syndrome = sum(
        int(support[6 - q]) * (2 ** j)
        for j, support in enumerate(supports)
    )

    # Syndrome value for a Z error on q
    z_syndrome = x_syndrome

    with qc.if_test((sx, x_syndrome)):
        qc.x(data[q])

    with qc.if_test((sz, z_syndrome)):
        qc.z(data[q])

# -----------------------------------------
# 6. Measure the corrected data qubits
# -----------------------------------------
qc.measure(data, out)

print("Steane code: syndrome extraction + correction")
print("Injected error: X on q0")
print("Qubit syndrome lookup:")

for q in range(7):
    value = sum(
        int(support[6 - q]) * (2 ** j)
        for j, support in enumerate(supports)
    )
    print(f"q{q}: syndrome integer = {value}")

print("\nRunning the circuit...")

simulator = AerSimulator()
result = simulator.run(qc, shots=1000).result()
counts = result.get_counts()

print("\nMeasurement results:")
for bitstring, count in counts.items():
    print(bitstring, ":", count)

print("\nCircuit completed.")

**Step 5 — Understand what the result means**

The circuit uses 13 qubits in total: seven data qubits and six ancillas.
| Component | Purpose |
|--------|--------|
| `data[0]` to `data[6]` | Store the encoded information |
| `anc[0]` to `anc[2]` | Measure the three \(Z\)\-type stabilizers |
| `anc[3]` to `anc[5]` | Measure the three \(X\)\-type stabilizers |
| `sx` and `sz` | Store the error syndromes |
| `if_test` | Apply corrections conditionally |
| `out` | Store the final data measurements |

Because we deliberately inserted an \(X\) error on \(q_0\), the first syndrome register should identify it, and the circuit should apply another \(X\) to that qubit to undo the error.

**One important detail:** the output includes several classical registers, so the printed bitstrings contain more than the seven final data bits. Don't interpret the entire string as the encoded answer.

**Step 6 — Checking whether the error was corrected**

In your Google Colab notebook, add a new code cell below the previous one. This code uses the counts variable created by your circuit.

In [ ]:

# Correct verification for the encoded logical-zero state

valid_codewords = {
    "0000000", "0011110",
    "0101101", "0110011",
    "1001011", "1010101",
    "1100110", "1111000"
}

valid_shots = 0
total_shots = sum(counts.values())

for bitstring, count in counts.items():
    final_data = bitstring.split()[0]

    if final_data in valid_codewords:
        valid_shots += count

rate = 100 * valid_shots / total_shots

print(f"Valid codeword outcomes: {valid_shots}/{total_shots}")
print(f"Code-space success rate: {rate:.2f}%")

if rate == 100:
    print("PASS: all measured outcomes are valid codewords.")
else:
    print("CHECK: some outcomes are outside the code space.")

**1.SO what does 100% result means**

every measured output belonged to the expected Steane-code space after the injected \(X\) error was corrected.

That's encouraging, but there's an important distinction:

- **Code-space success:** the result satisfies the code's computational-basis pattern.
- **Error-correction success:** the circuit identifies the error, corrects it, and preserves the original logical information.

The second is what we'll test now. We'll deliberately inject each of the 21 possible single-qubit Pauli errors and verify its syndrome and final code-space membership.

**2. Test all 21 errors automatically**

Open a **new Colab cell**. Paste and run this entire code cell. It builds a fresh circuit for each error, so the tests won't interfere with one another.

In [ ]:

import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator

# -----------------------------------------
# Steane-code test: all 21 single-qubit errors
# -----------------------------------------

codewords = {
    "0000000", "0011110",
    "0101101", "0110011",
    "1001011", "1010101",
    "1100110", "1111000"
}

supports = ["0011110", "0101101", "1001011"]

amplitudes = np.zeros(128, dtype=complex)
for word in codewords:
    amplitudes[int(word, 2)] = 1 / np.sqrt(8)

circuits = []
test_names = []
expected_syndromes = []

for error_type in ["X", "Y", "Z"]:
    for error_qubit in range(7):

        data = QuantumRegister(7, "data")
        anc = QuantumRegister(6, "anc")
        sx = ClassicalRegister(3, "sx")
        sz = ClassicalRegister(3, "sz")
        out = ClassicalRegister(7, "out")

        qc = QuantumCircuit(data, anc, sx, sz, out)
        qc.initialize(amplitudes, list(data))

        # Inject one error
        getattr(qc, error_type.lower())(data[error_qubit])

        # Measure Z stabilizers: detect X components
        for j, support in enumerate(supports):
            for q in range(7):
                if support[6 - q] == "1":
                    qc.cx(data[q], anc[j])
            qc.measure(anc[j], sx[j])

        # Measure X stabilizers: detect Z components
        for j, support in enumerate(supports):
            a = anc[3 + j]
            qc.h(a)
            for q in range(7):
                if support[6 - q] == "1":
                    qc.cx(a, data[q])
            qc.h(a)
            qc.measure(a, sz[j])

        # Calculate expected syndrome for this qubit
        syndrome = sum(
            int(support[6 - error_qubit]) * (2 ** j)
            for j, support in enumerate(supports)
        )

        # Correct X component and Z component
        if error_type in ["X", "Y"]:
            with qc.if_test((sx, syndrome)):
                qc.x(data[error_qubit])

        if error_type in ["Z", "Y"]:
            with qc.if_test((sz, syndrome)):
                qc.z(data[error_qubit])

        qc.measure(data, out)

        circuits.append(qc)
        test_names.append(f"{error_type} on q{error_qubit}")

        expected_syndromes.append((
            syndrome if error_type in ["X", "Y"] else 0,
            syndrome if error_type in ["Z", "Y"] else 0
        ))

# Run all 21 circuits
simulator = AerSimulator()
result = simulator.run(circuits, shots=100).result()

passed = 0
failed_tests = []

print("STEANE CODE: 21 SINGLE-ERROR TESTS\n")

for i, name in enumerate(test_names):
    counts = result.get_counts(i)
    expected_sx, expected_sz = expected_syndromes[i]
    correct_shots = 0

    for bitstring, count in counts.items():
        parts = bitstring.split()

        # Registers display in reverse creation order:
        # out, sz, sx
        if len(parts) != 3:
            continue

        final_data = parts[0]
        measured_sz = int(parts[1], 2)
        measured_sx = int(parts[2], 2)

        syndrome_ok = (
            measured_sx == expected_sx
            and measured_sz == expected_sz
        )
        codeword_ok = final_data in codewords

        if syndrome_ok and codeword_ok:
            correct_shots += count

    success = correct_shots == 100

    if success:
        passed += 1
        status = "PASS"
    else:
        status = "CHECK"
        failed_tests.append(name)

    print(f"{name:10s}: {status} ({correct_shots}/100 shots)")

print("\n--------------------------------")
print(f"Tests passed: {passed}/21")
print(f"Tests needing investigation: {21-passed}")
print("--------------------------------")

if passed == 21:
    print("All single-qubit Pauli error tests passed!")
else:
    print("Inspect the CHECK results before proceeding.")
    print("Tests to investigate:", failed_tests)

**3. How to interpret results**

***Expected ideal-simulation result: 21/21 PASS***

Each error should produce the expected syndrome, and all 100 shots should finish with valid codewords. A CHECK result means we need to inspect the syndrome ordering, correction logic, or output-register parsing.

One limitation remains: these checks validate the expected syndrome and code-space membership, but computational-basis measurements alone cannot prove preservation of every superposition or phase. We'll address that in a later test.

**4.Next step**

we get 21/21, we'll move on to the exciting part: adding a noise model with Qiskit Aer and comparing logical error rates with an unprotected qubit.

**5.Testing code under noise**

In [ ]:

from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit import transpile
from qiskit_aer import AerSimulator

# Create a simple depolarizing noise model
noise_model = NoiseModel()

noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.001, 1), ["x", "h"]
)

noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.005, 2), ["cx"]
)

noisy_simulator = AerSimulator(noise_model=noise_model)

# Convert the existing 21 circuits into supported gates
compiled_circuits = transpile(
    circuits,
    noisy_simulator,
    optimization_level=0
)

# Run with noise
noisy_result = noisy_simulator.run(
    compiled_circuits,
    shots=1000
).result()

print("Noisy simulation completed!")
print("Number of error tests:", len(circuits))
print("Shots per test:", 1000)
print("Noise model: depolarizing errors on X, H and CX gates")

**Step 7-Measure performance under noise**

Adding a new cell below your current one and run this code. It uses a noise model attached to the simulator's u and cx gates, making it easier to ensure the noise is applied to the transpiled circui

In [ ]:

from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer import AerSimulator
from qiskit import transpile
import matplotlib.pyplot as plt

# Use noise on gates that appear in the compiled circuit
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.001, 1), ["u"]
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.005, 2), ["cx"]
)

noisy_simulator = AerSimulator(noise_model=noise_model)

# Compile the 21 circuits into the simulator's gate basis
compiled_noisy = transpile(
    circuits,
    basis_gates=["u", "cx"],
    optimization_level=0
)

print("Compiled gate types:", compiled_noisy[0].count_ops())

# Run the circuits with noise
noisy_result = noisy_simulator.run(
    compiled_noisy, shots=1000
).result()

# Check whether the final measurements belong to the code space
valid_codewords = {
    "0000000", "0011110",
    "0101101", "0110011",
    "1001011", "1010101",
    "1100110", "1111000"
}

success_rates = []

for i, name in enumerate(test_names):
    counts = noisy_result.get_counts(i)
    total = sum(counts.values())
    valid = 0

    for bitstring, count in counts.items():
        final_data = bitstring.split()[0]
        if final_data in valid_codewords:
            valid += count

    rate = 100 * valid / total
    success_rates.append(rate)
    print(f"{name:10s}: {rate:.2f}% in code space")

# Plot the 21 test results
plt.figure(figsize=(12, 5))
plt.bar(test_names, success_rates)
plt.axhline(100, linestyle="--")
plt.ylabel("Valid code-space outcomes (%)")
plt.xlabel("Injected error")
plt.title("Steane Code Under Simulated Gate Noise")
plt.xticks(rotation=60)
plt.ylim(0, 105)
plt.tight_layout()
plt.show()

**Interpreting hthe graph**
- 100% means every measured output was in the expected code space.

- A lower percentage means some outputs ended outside that code space.

- This is a useful diagnostic, but it is **not by itself the logical error rate:** a wrong logical state can still be inside the code space.

Our next experiment will compare protected and unprotected qubits under equivalent noise, using a logical-state recovery metric. That's the stronger evidence that your code actually protects information.

**Step 8 — Check whether the error was corrected**

In your Google Colab notebook, add a new code cell below the previous one. This code uses the counts variable created by your circuit.

In [ ]:

# Verify the results from the previous circuit

print("Most common measurement outcomes:")
for bitstring, count in sorted(
    counts.items(), key=lambda item: item[1], reverse=True
)[:10]:
    print(bitstring, ":", count)

# Qiskit displays classical registers in reverse order.
# Our registers were created as sx, sz, out.
print("\nChecking the final data measurements...")

corrected_shots = 0
total_shots = sum(counts.values())

for bitstring, count in counts.items():
    registers = bitstring.split()

    # The 'out' register is the first group displayed.
    final_data = registers[0]

    if final_data == "0000000":
        corrected_shots += count

success_rate = 100 * corrected_shots / total_shots

print("Total shots:", total_shots)
print("Shots ending in logical-zero basis state:", corrected_shots)
print(f"Final all-zero measurement rate: {success_rate:.2f}%")

**Important correction:**

your encoded logical-zero state is a superposition of eight valid bitstrings, not just **0000000**. So the correct success test is whether the final measurement belongs to the codeword set.

Use this cell instead of the all-zero check above:

In [ ]:

# Correct verification for the encoded logical-zero state

valid_codewords = {
    "0000000", "0011110",
    "0101101", "0110011",
    "1001011", "1010101",
    "1100110", "1111000"
}

valid_shots = 0
total_shots = sum(counts.values())

for bitstring, count in counts.items():
    final_data = bitstring.split()[0]

    if final_data in valid_codewords:
        valid_shots += count

rate = 100 * valid_shots / total_shots

print(f"Valid codeword outcomes: {valid_shots}/{total_shots}")
print(f"Code-space success rate: {rate:.2f}%")

if rate == 100:
    print("PASS: all measured outcomes are valid codewords.")
else:
    print("CHECK: some outcomes are outside the code space.")

**What 100% result means**
  
  test passed: every measured output belonged to the expected Steane-code space after the injected \(X\) error was corrected.

  important distinction:
  
-  **Code-space success:** the result satisfies the code's computational-basis pattern.

- **Error-correction success:** the circuit identifies the error, on correcting and preserving the original logical information.

The second thing to be test is to deliberately inject each of the 21 possible single-qubit Pauli errors and verifying its syndrome and final code-space membership.

**2. Testing all 21 errors automatically**



In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator

# -----------------------------------------
# Steane-code test: all 21 single-qubit errors
# -----------------------------------------

codewords = {
    "0000000", "0011110",
    "0101101", "0110011",
    "1001011", "1010101",
    "1100110", "1111000"
}

supports = ["0011110", "0101101", "1001011"]

amplitudes = np.zeros(128, dtype=complex)
for word in codewords:
    amplitudes[int(word, 2)] = 1 / np.sqrt(8)

**Testing the code under noise**



In [ ]:

from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit import transpile
from qiskit_aer import AerSimulator

# Create a simple depolarizing noise model
noise_model = NoiseModel()

noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.001, 1), ["x", "h"]
)

noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.005, 2), ["cx"]
)

noisy_simulator = AerSimulator(noise_model=noise_model)

# Convert the existing 21 circuits into supported gates
compiled_circuits = transpile(
    circuits,
    noisy_simulator,
    optimization_level=0
)

# Run with noise
noisy_result = noisy_simulator.run(
    compiled_circuits,
    shots=1000
).result()

print("Noisy simulation completed!")
print("Number of error tests:", len(circuits))
print("Shots per test:", 1000)
print("Noise model: depolarizing errors on X, H and CX gates")

**Measuring performance under noise**



In [ ]:

from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer import AerSimulator
from qiskit import transpile
import matplotlib.pyplot as plt

# Use noise on gates that appear in the compiled circuit
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.001, 1), ["u"]
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.005, 2), ["cx"]
)

noisy_simulator = AerSimulator(noise_model=noise_model)

# Compile the 21 circuits into the simulator's gate basis
compiled_noisy = transpile(
    circuits,
    basis_gates=["u", "cx"],
    optimization_level=0
)

print("Compiled gate types:", compiled_noisy[0].count_ops())

# Run the circuits with noise
noisy_result = noisy_simulator.run(
    compiled_noisy, shots=1000
).result()

# Check whether the final measurements belong to the code space
valid_codewords = {
    "0000000", "0011110",
    "0101101", "0110011",
    "1001011", "1010101",
    "1100110", "1111000"
}

success_rates = []

for i, name in enumerate(test_names):
    counts = noisy_result.get_counts(i)
    total = sum(counts.values())
    valid = 0

    for bitstring, count in counts.items():
        final_data = bitstring.split()[0]
        if final_data in valid_codewords:
            valid += count

    rate = 100 * valid / total
    success_rates.append(rate)
    print(f"{name:10s}: {rate:.2f}% in code space")

# Plot the 21 test results
plt.figure(figsize=(12, 5))
plt.bar(test_names, success_rates)
plt.axhline(100, linestyle="--")
plt.ylabel("Valid code-space outcomes (%)")
plt.xlabel("Injected error")
plt.title("Steane Code Under Simulated Gate Noise")
plt.xticks(rotation=60)
plt.ylim(0, 105)
plt.tight_layout()
plt.show()

- 100% means every measured output was in the expected code space.
- A lower percentage means some outputs ended outside that code space.
- This is a useful diagnostic, but it is not by itself the logical error rate: a wrong logical state can still be inside the code space.

**Make a clear results table and graph:**

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# success_rates was calculated by your previous cell
# Order: X on q0-q6, Y on q0-q6, Z on q0-q6

x_rates = success_rates[0:7]
y_rates = success_rates[7:14]
z_rates = success_rates[14:21]

groups = {
    "X errors": x_rates,
    "Y errors": y_rates,
    "Z errors": z_rates
}

print("NOISY STEANE-CODE RESULTS")
print("-" * 40)

for name, values in groups.items():
    print(
        f"{name}: mean = {np.mean(values):.2f}%, "
        f"minimum = {np.min(values):.2f}%, "
        f"maximum = {np.max(values):.2f}%"
    )

# Plot average code-space rates by error type
labels = list(groups.keys())
means = [np.mean(values) for values in groups.values()]

plt.figure(figsize=(8, 5))
plt.bar(labels, means)
plt.ylabel("Average valid code-space outcomes (%)")
plt.title("Steane Code: Results Under Simulated Gate Noise")
plt.ylim(0, 100)

for i, value in enumerate(means):
    plt.text(i, value + 1, f"{value:.1f}%", ha="center")

plt.tight_layout()
plt.show()

Generating a graph showing the difference between

 **Steane code vs phyical unprotected qubit:**

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# STEANE CODE: CONTROLLED NOISE BENCHMARK
# ==========================================
# Same independent depolarizing Pauli noise per
# physical qubit. Syndrome extraction/recovery
# is assumed PERFECT in this benchmark.

rng = np.random.default_rng(42)

# Three Hamming-code parity-check supports.
# Bit q is selected by support[6-q].
supports = ["0011110", "0101101", "1001011"]

# Build the six stabilizer generators as binary
# vectors [x0...x6, z0...z6].
stabilizers = []

for support in supports:
    bits = np.array(
        [int(support[6-q]) for q in range(7)],
        dtype=np.uint8
    )

    # X-type stabilizer
    stabilizers.append(np.concatenate([bits, np.zeros(7, dtype=np.uint8)]))

    # Z-type stabilizer
    stabilizers.append(np.concatenate([np.zeros(7, dtype=np.uint8), bits]))

# Enumerate all 64 stabilizer products.
stabilizer_set = set()

for mask in range(64):
    v = np.zeros(14, dtype=np.uint8)
    for j in range(6):
        if (mask >> j) & 1:
            v ^= stabilizers[j]
    stabilizer_set.add(tuple(v.tolist()))

# Map each nonzero syndrome to its physical qubit.
syndrome_to_qubit = {}

for q in range(7):
    s = sum(
        int(support[6-q]) * (2**j)
        for j, support in enumerate(supports)
    )
    syndrome_to_qubit[s] = q

def steane_logical_failure(p, shots=30000):
    failures = 0

    for _ in range(shots):
        x = np.zeros(7, dtype=np.uint8)
        z = np.zeros(7, dtype=np.uint8)

        # Independent depolarizing channel:
        # I with probability 1-p; X,Y,Z each p/3.
        for q in range(7):
            r = rng.random()
            if r < p/3:
                x[q] = 1              # X
            elif r < 2*p/3:
                z[q] = 1              # Z
            elif r < p:
                x[q] = z[q] = 1       # Y

        # Syndrome from the three Z checks (X component)
        sx = [
            int(np.dot(
                np.array([int(s[6-q]) for q in range(7)]),
                x
            ) % 2)
            for s in supports
        ]

        # Syndrome from the three X checks (Z component)
        sz = [
            int(np.dot(
                np.array([int(s[6-q]) for q in range(7)]),
                z
            ) % 2)
            for s in supports
        ]

        sx_value = sum(bit * (2**j) for j, bit in enumerate(sx))
        sz_value = sum(bit * (2**j) for j, bit in enumerate(sz))

        # Perfect minimum-weight single-qubit correction
        if sx_value:
            x[syndrome_to_qubit[sx_value]] ^= 1
        if sz_value:
            z[syndrome_to_qubit[sz_value]] ^= 1

        # Residual error is harmless iff it is a stabilizer
        residual = tuple(np.concatenate([x, z]).tolist())

        if residual not in stabilizer_set:
            failures += 1

    return failures / shots

# Sweep several physical error probabilities
p_values = [0.001, 0.003, 0.005, 0.01, 0.02, 0.03, 0.05]
shots = 30000

unprotected = []
protected = []

for p in p_values:
    unprotected.append(p)
    protected.append(steane_logical_failure(p, shots))

print("Physical error | Unprotected error | Steane logical error")
print("-" * 59)

for p, u, l in zip(p_values, unprotected, protected):
    print(f"{p:13.3%} | {u:16.3%} | {l:19.3%}")

plt.figure(figsize=(8, 5))
plt.plot(
    np.array(p_values)*100,
    np.array(unprotected)*100,
    marker="o",
    label="Unprotected physical qubit"
)
plt.plot(
    np.array(p_values)*100,
    np.array(protected)*100,
    marker="s",
    label="Steane logical qubit (perfect recovery)"
)
plt.xlabel("Physical Pauli-error probability per qubit (%)")
plt.ylabel("Error probability (%)")
plt.title("Steane Code vs Unprotected Qubit")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Showing the exponential improvement from steane error correction**

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Use results from your completed benchmark
results = pd.DataFrame({
    "Physical error probability (%)":
        [100 * p for p in p_values],
    "Unprotected error (%)":
        [100 * p for p in unprotected],
    "Steane logical failure (%)":
        [100 * p for p in protected]
})

results["Relative reduction (%)"] = (
    100 * (
        results["Unprotected error (%)"]
        - results["Steane logical failure (%)"]
    ) / results["Unprotected error (%)"]
)

print("STEANE CODE PERFORMANCE")
display(results.round(3))

plt.figure(figsize=(8, 5))
plt.plot(
    results["Physical error probability (%)"],
    results["Relative reduction (%)"],
    marker="o"
)
plt.xlabel("Physical error probability (%)")
plt.ylabel("Logical error reduction (%)")
plt.title("Improvement from Steane Error Correction")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

We'll implement a two-level concatenated Steane [[7,1,3]] code with 49 physical qubits, independent depolarizing noise at p = 5% and perfect recovery at both levels.

The key is to decode each group of 7 physical qubits first, then decode the resulting 7 level-one logical qubits. We'll compare the final logical failure rate with the unprotected physical error rate using

PL R = 1 × 100%. 0.05

We'll use Monte Carlo sampling for the 49-qubit physical noise, and report statistical uncertainty so we don't mistake simulation noise for an improvement.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# TWO-LEVEL CONCATENATED STEANE CODE: 49 PHYSICAL QUBITS
# Noise: independent depolarizing Pauli channel
# Recovery: perfect at both levels
# =========================================================

P = 0.05
SHOTS = 100_000
SEED = 2026

rng = np.random.default_rng(SEED)

# Three parity-check rows for the Steane [[7,1,3]] code
supports = ["0011110", "0101101", "1001011"]

H = np.array([
    [int(s[6-q]) for q in range(7)]
    for s in supports
], dtype=np.uint8)

# Map each nonzero syndrome to its qubit
syndrome_to_qubit = {}

for q in range(7):
    syndrome = sum(
        int(H[j, q]) * (2**j) for j in range(3)
    )
    syndrome_to_qubit[syndrome] = q

# Construct the six stabilizer generators
stabilizers = []

for row in H:
    stabilizers.append(
        np.concatenate([row, np.zeros(7, dtype=np.uint8)])
    )
    stabilizers.append(
        np.concatenate([np.zeros(7, dtype=np.uint8), row])
    )

# Enumerate all 64 stabilizer products
stabilizer_set = set()

for mask in range(64):
    v = np.zeros(14, dtype=np.uint8)

    for j in range(6):
        if (mask >> j) & 1:
            v ^= stabilizers[j]

    stabilizer_set.add(tuple(v.tolist()))


def decode_steane(x, z):
    """
    Perfect CSS recovery for one Steane block.
    x and z are binary error vectors of length 7.
    """

    sx = H @ x % 2
    sz = H @ z % 2

    sx_value = sum(
        int(bit) * (2**j) for j, bit in enumerate(sx)
    )
    sz_value = sum(
        int(bit) * (2**j) for j, bit in enumerate(sz)
    )

    # Correct X errors using the Z-check syndrome
    if sx_value:
        x[syndrome_to_qubit[sx_value]] ^= 1

    # Correct Z errors using the X-check syndrome
    if sz_value:
        z[syndrome_to_qubit[sz_value]] ^= 1

    return x, z


def concatenated_steane_failure(p, shots=SHOTS):
    failures = 0

    for _ in range(shots):

        # Seven inner blocks, each containing seven qubits
        # 0 = I, 1 = X, 2 = Y, 3 = Z
        r = rng.random((7, 7))

        physical_errors = np.zeros((7, 7), dtype=np.uint8)
        physical_errors[r < p/3] = 1
        physical_errors[(r >= p/3) & (r < 2*p/3)] = 2
        physical_errors[(r >= 2*p/3) & (r < p)] = 3

        # Effective errors on the seven inner logical qubits
        logical_x = np.zeros(7, dtype=np.uint8)
        logical_z = np.zeros(7, dtype=np.uint8)

        # LEVEL 1: decode each of the seven inner blocks
        for block in range(7):

            e = physical_errors[block]

            x = ((e == 1) | (e == 2)).astype(np.uint8)
            z = ((e == 2) | (e == 3)).astype(np.uint8)

            x, z = decode_steane(x, z)

            # Residual zero-syndrome errors are either
            # stabilizers or logical Pauli errors.
            logical_x[block] = np.sum(x) % 2
            logical_z[block] = np.sum(z) % 2

        # LEVEL 2: decode the seven effective logical qubits
        logical_x, logical_z = decode_steane(
            logical_x, logical_z
        )

        residual = tuple(
            np.concatenate([logical_x, logical_z]).tolist()
        )

        # Failure occurs if the final error is not a stabilizer
        if residual not in stabilizer_set:
            failures += 1

    p_logical = failures / shots

    # Approximate binomial standard error
    standard_error = np.sqrt(
        p_logical * (1 - p_logical) / shots
    )

    return failures, p_logical, standard_error


# =========================================================
# RUN THE BENCHMARK
# =========================================================

failures, p_logical, se = concatenated_steane_failure(P)

relative_reduction = (1 - p_logical / P) * 100

# Approximate 95% confidence interval
lower = max(0, p_logical - 1.96 * se)
upper = min(1, p_logical + 1.96 * se)

reduction_lower = (1 - upper / P) * 100
reduction_upper = (1 - lower / P) * 100

print("TWO-LEVEL CONCATENATED STEANE BENCHMARK")
print("=" * 48)
print(f"Physical qubits             : 49")
print(f"Physical error probability  : {P:.2%}")
print(f"Monte Carlo trials          : {SHOTS:,}")
print(f"Logical failures            : {failures:,}")
print(f"Logical error probability   : {p_logical:.4%}")
print(f"Approx. 95% CI for P_L      : "
      f"[{lower:.4%}, {upper:.4%}]")
print(f"Relative error reduction    : {relative_reduction:.2f}%")
print(f"Approx. 95% CI for reduction: "
      f"[{reduction_lower:.2f}%, {reduction_upper:.2f}%]")

if reduction_lower > 50:
    print("RESULT: Reduction above 50% is supported "
          "by the approximate 95% interval.")
elif relative_reduction > 50:
    print("RESULT: Point estimate exceeds 50%, "
          "but the confidence interval should be checked.")
else:
    print("RESULT: The 50% reduction target was not reached.")


# =========================================================
# PLOT THE COMPARISON AT 5% PHYSICAL ERROR
# =========================================================

plt.figure(figsize=(8, 5))

labels = [
    "Unprotected\nphysical qubit",
    "Two-level\nSteane code"
]
values = [P * 100, p_logical * 100]

plt.bar(labels, values)
plt.ylabel("Error probability (%)")
plt.title("49-Qubit Concatenated Steane Code at 5% Noise")
plt.grid(axis="y", alpha=0.3)

for i, value in enumerate(values):
    plt.text(i, value + 0.05, f"{value:.3f}%", ha="center")

plt.tight_layout()
plt.show()

# Relative reduction plot
plt.figure(figsize=(6, 4))
plt.bar(["Relative error reduction"], [relative_reduction])
plt.axhline(50, linestyle="--", label="50% target")
plt.ylabel("Reduction (%)")
plt.ylim(0, 100)
plt.title("Concatenated Steane Code Performance")
plt.legend()
plt.tight_layout()
plt.show()

Physical error probability: 5.00%

Logical failures: 1,470 out of 100,000

Logical error probability: 1.47%

Relative error reduction: 70.60%

Approximate 95% confidence interval for reduction: 69.11% to 72.09%

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# FAIR COMPARISON: SINGLE-LEVEL vs TWO-LEVEL STEANE CODE
# Physical depolarizing probability = 5%
# Perfect recovery at every level
# ==========================================================

P = 0.05
SHOTS = 100_000
SEED = 2026

rng = np.random.default_rng(SEED)

# Hamming parity-check matrix
supports = ["0011110", "0101101", "1001011"]

H = np.array([
    [int(s[6-q]) for q in range(7)]
    for s in supports
], dtype=np.uint8)

syndrome_to_qubit = {
    sum(int(H[j, q]) * (2**j) for j in range(3)): q
    for q in range(7)
}

# Six Steane stabilizer generators: [x | z]
stabilizers = []

for row in H:
    stabilizers.append(
        np.concatenate([row, np.zeros(7, dtype=np.uint8)])
    )
    stabilizers.append(
        np.concatenate([np.zeros(7, dtype=np.uint8), row])
    )

# All 64 stabilizer products
stabilizer_set = set()

for mask in range(64):
    v = np.zeros(14, dtype=np.uint8)
    for j in range(6):
        if (mask >> j) & 1:
            v ^= stabilizers[j]
    stabilizer_set.add(tuple(v.tolist()))


def decode_steane(x, z):
    """Perfect minimum-weight CSS recovery on one 7-qubit block."""
    x = x.copy()
    z = z.copy()

    sx = H @ x % 2
    sz = H @ z % 2

    sx_value = sum(int(b) * (2**j) for j, b in enumerate(sx))
    sz_value = sum(int(b) * (2**j) for j, b in enumerate(sz))

    if sx_value:
        x[syndrome_to_qubit[sx_value]] ^= 1

    if sz_value:
        z[syndrome_to_qubit[sz_value]] ^= 1

    # Verify that recovery has produced zero syndrome.
    assert np.all(H @ x % 2 == 0)
    assert np.all(H @ z % 2 == 0)

    return x, z


def is_stabilizer(x, z):
    residual = tuple(np.concatenate([x, z]).tolist())
    return residual in stabilizer_set


def sample_pauli_errors(shape, p):
    """
    I: 1-p; X, Y, Z: p/3 each.
    Returns integer labels: I=0, X=1, Y=2, Z=3.
    """
    r = rng.random(shape)
    e = np.zeros(shape, dtype=np.uint8)
    e[(r < p/3)] = 1
    e[(r >= p/3) & (r < 2*p/3)] = 2
    e[(r >= 2*p/3) & (r < p)] = 3
    return e


def pauli_to_xz(e):
    x = ((e == 1) | (e == 2)).astype(np.uint8)
    z = ((e == 2) | (e == 3)).astype(np.uint8)
    return x, z


def single_level_benchmark(p, shots):
    failures = 0

    for _ in range(shots):
        e = sample_pauli_errors((7,), p)
        x, z = pauli_to_xz(e)

        x, z = decode_steane(x, z)

        if not is_stabilizer(x, z):
            failures += 1

    return failures


def two_level_benchmark(p, shots):
    failures = 0

    for _ in range(shots):
        # Seven inner blocks, seven physical qubits per block.
        e = sample_pauli_errors((7, 7), p)

        logical_x = np.zeros(7, dtype=np.uint8)
        logical_z = np.zeros(7, dtype=np.uint8)

        # Inner decoding
        for block in range(7):
            x, z = pauli_to_xz(e[block])
            x, z = decode_steane(x, z)

            # Residual normalizer: parity identifies logical class.
            logical_x[block] = np.sum(x) % 2
            logical_z[block] = np.sum(z) % 2

        # Outer decoding
        logical_x, logical_z = decode_steane(
            logical_x, logical_z
        )

        if not is_stabilizer(logical_x, logical_z):
            failures += 1

    return failures


def summarize(name, failures, shots, p):
    rate = failures / shots
    se = np.sqrt(rate * (1-rate) / shots)

    # Approximate 95% confidence interval for the logical rate
    low = max(0, rate - 1.96 * se)
    high = min(1, rate + 1.96 * se)

    reduction = (1 - rate/p) * 100
    reduction_low = (1 - high/p) * 100
    reduction_high = (1 - low/p) * 100

    return {
        "name": name,
        "failures": failures,
        "rate": rate,
        "low": low,
        "high": high,
        "reduction": reduction,
        "reduction_low": reduction_low,
        "reduction_high": reduction_high,
    }


# Run each benchmark using the same number of trials.
single_failures = single_level_benchmark(P, SHOTS)
double_failures = two_level_benchmark(P, SHOTS)

single = summarize(
    "Single-level Steane (7 qubits)",
    single_failures, SHOTS, P
)

double = summarize(
    "Two-level Steane (49 qubits)",
    double_failures, SHOTS, P
)

# ==========================================================
# RESULTS
# ==========================================================

print("STEANE CODE BENCHMARK AT 5% PHYSICAL NOISE")
print("=" * 76)
print(f"Trials per method: {SHOTS:,}")
print(f"Unprotected physical error rate: {P:.3%}\n")

for result in [single, double]:
    print(result["name"])
    print(f"  Failures: {result['failures']:,} / {SHOTS:,}")
    print(f"  Logical error: {result['rate']:.4%}")
    print(f"  Approx. 95% CI: "
          f"[{result['low']:.4%}, {result['high']:.4%}]")
    print(f"  Relative reduction: {result['reduction']:.2f}%")
    print(f"  Approx. 95% CI for reduction: "
          f"[{result['reduction_low']:.2f}%, "
          f"{result['reduction_high']:.2f}%]\n")

improvement = single["rate"] - double["rate"]
print(f"Absolute logical-error decrease: {improvement:.4%}")
print(f"Relative decrease vs single-level: "
      f"{100*improvement/single['rate']:.2f}%"
      if single["rate"] > 0 else "Single-level rate is zero.")

if double["reduction_low"] > 50:
    print("\nTARGET: >50% reduction supported by the "
          "approximate 95% interval.")
elif double["reduction"] > 50:
    print("\nTARGET: Point estimate exceeds 50%, but uncertainty "
          "does not fully establish it.")
else:
    print("\nTARGET: >50% reduction not demonstrated.")


# ==========================================================
# PLOT 1: ERROR PROBABILITIES
# ==========================================================

labels = [
    "Unprotected\nphysical qubit",
    "Single-level\nSteane (7)",
    "Two-level\nSteane (49)"
]

rates = [P, single["rate"], double["rate"]]

plt.figure(figsize=(9, 5))
bars = plt.bar(labels, np.array(rates) * 100)

plt.ylabel("Error probability (%)")
plt.title("Steane Code Comparison at 5% Physical Noise")
plt.grid(axis="y", alpha=0.3)

for bar, rate in zip(bars, rates):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.04,
        f"{rate*100:.3f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()


# ==========================================================
# PLOT 2: RELATIVE ERROR REDUCTION
# ==========================================================

reductions = [0, single["reduction"], double["reduction"]]

plt.figure(figsize=(9, 5))
bars = plt.bar(labels, reductions)
plt.axhline(
    50, linestyle="--",
    label="50% target"
)

plt.ylabel("Relative error reduction (%)")
plt.title("Relative Error Reduction vs Unprotected Qubit")
plt.ylim(min(-5, min(reductions)-5), 105)
plt.grid(axis="y", alpha=0.3)
plt.legend()

for bar, value in zip(bars, reductions):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 1,
        f"{value:.2f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

Important: this is a comparison of ideal decoding performance, not a complete fault-tolerant circuit simulation. Perfect recovery hides the errors that real syndrome-extraction circuits would introduce

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# STEANE CODE PERFORMANCE: 7 vs 49 PHYSICAL QUBITS
# Independent depolarizing noise; perfect recovery
# ==========================================================

P_VALUES = [0.001, 0.003, 0.005, 0.01, 0.02, 0.03, 0.05]
SHOTS = 20_000
rng = np.random.default_rng(2026)

H = np.array([
    [0,1,1,1,1,0,0],
    [1,0,1,1,0,1,0],
    [1,1,0,1,0,0,1]
], dtype=np.uint8)

syndrome_to_qubit = {
    sum(int(H[j, q]) * (2**j) for j in range(3)): q
    for q in range(7)
}

stabilizers = []
for row in H:
    stabilizers.append(np.r_[row, np.zeros(7, dtype=np.uint8)])
    stabilizers.append(np.r_[np.zeros(7, dtype=np.uint8), row])

stabilizer_set = set()
for mask in range(64):
    v = np.zeros(14, dtype=np.uint8)
    for j in range(6):
        if (mask >> j) & 1:
            v ^= stabilizers[j]
    stabilizer_set.add(tuple(v.tolist()))


def decode(x, z):
    x, z = x.copy(), z.copy()
    sx = H @ x % 2
    sz = H @ z % 2

    a = sum(int(b) * (2**j) for j, b in enumerate(sx))
    b = sum(int(b) * (2**j) for j, b in enumerate(sz))

    if a:
        x[syndrome_to_qubit[a]] ^= 1
    if b:
        z[syndrome_to_qubit[b]] ^= 1

    return x, z


def is_stabilizer(x, z):
    return tuple(np.r_[x, z].tolist()) in stabilizer_set


def sample_errors(shape, p):
    r = rng.random(shape)
    e = np.zeros(shape, dtype=np.uint8)
    e[(r < p/3)] = 1              # X
    e[(r >= p/3) & (r < 2*p/3)] = 2  # Y
    e[(r >= 2*p/3) & (r < p)] = 3    # Z
    return e


def to_xz(e):
    return ((e == 1) | (e == 2)).astype(np.uint8), \
           ((e == 2) | (e == 3)).astype(np.uint8)


def single_level(p, shots):
    failures = 0
    for _ in range(shots):
        e = sample_errors((7,), p)
        x, z = to_xz(e)
        x, z = decode(x, z)
        failures += not is_stabilizer(x, z)
    return failures


def two_level(p, shots):
    failures = 0
    for _ in range(shots):
        e = sample_errors((7, 7), p)
        lx = np.zeros(7, dtype=np.uint8)
        lz = np.zeros(7, dtype=np.uint8)

        # Decode seven inner blocks
        for block in range(7):
            x, z = to_xz(e[block])
            x, z = decode(x, z)
            lx[block] = np.sum(x) % 2
            lz[block] = np.sum(z) % 2

        # Decode outer block
        lx, lz = decode(lx, lz)
        failures += not is_stabilizer(lx, lz)

    return failures


# ==========================================================
# RUN BENCHMARKS
# ==========================================================

rows = []

for p in P_VALUES:
    f1 = single_level(p, SHOTS)
    f2 = two_level(p, SHOTS)

    l1 = f1 / SHOTS
    l2 = f2 / SHOTS

    rows.append({
        "Physical error probability (%)": p * 100,
        "Unprotected error (%)": p * 100,
        "Steane logical failure (%)": l1 * 100,
        "Single-level reduction (%)": (1 - l1/p) * 100,
        "49-qubit logical failure (%)": l2 * 100,
        "Two-level reduction (%)": (1 - l2/p) * 100,
        "Single-level failures": f1,
        "Two-level failures": f2
    })

results = pd.DataFrame(rows)

print("STEANE CODE PERFORMANCE")
print(f"Trials per physical error rate: {SHOTS:,}")
display(results.round(3))


# ==========================================================
# GRAPH 1: ERROR PROBABILITY COMPARISON
# ==========================================================

x = results["Physical error probability (%)"]

plt.figure(figsize=(9, 5))
plt.plot(
    x, results["Unprotected error (%)"],
    marker="o", label="Unprotected qubit"
)
plt.plot(
    x, results["Steane logical failure (%)"],
    marker="s", label="Single-level Steane (7 qubits)"
)
plt.plot(
    x, results["49-qubit logical failure (%)"],
    marker="^", label="Two-level Steane (49 qubits)"
)

plt.xlabel("Physical error probability (%)")
plt.ylabel("Logical/error probability (%)")
plt.title("Steane Code Performance Comparison")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# ==========================================================
# GRAPH 2: RELATIVE ERROR REDUCTION
# ==========================================================

plt.figure(figsize=(9, 5))
plt.plot(
    x, results["Single-level reduction (%)"],
    marker="s", label="Single-level Steane"
)
plt.plot(
    x, results["Two-level reduction (%)"],
    marker="^", label="Two-level Steane"
)
plt.axhline(
    50, linestyle="--", label="50% reduction target"
)

plt.xlabel("Physical error probability (%)")
plt.ylabel("Relative error reduction (%)")
plt.title("Improvement from Steane Error Correction")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# ==========================================================
# FOCUS RESULT: 5% PHYSICAL NOISE
# ==========================================================

r5 = results.iloc[-1]

print("\nRESULT AT 5% PHYSICAL ERROR")
print("-" * 42)
print(f"Single-level reduction: {r5['Single-level reduction (%)']:.2f}%")
print(f"Two-level reduction:    {r5['Two-level reduction (%)']:.2f}%")

if r5["Two-level reduction (%)"] > 50:
    print("Observed two-level reduction exceeds 50%.")
else:
    print("Observed two-level reduction does not exceed 50%.")

print("Monte Carlo estimates; finite sampling uncertainty applies.")